In [1]:
import subprocess, sys

subprocess.run(["nvidia-smi"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "transformers", "--quiet"
], check=True)

print("✓ Packages ready")

Mon Mar  9 15:34:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

WORK_DIR = "/kaggle/working"

BASE = "/kaggle/input/datasets/rishta123/vaccine-meme-dataset"

TRAIN_IMGDIR    = f"{BASE}/train-20260302T182945Z-3-001/train/train_images"
TRAIN_LABEL_CSV = f"{BASE}/train-20260302T182945Z-3-001/train/index_label_train.csv"
TRAIN_TEXT_CSV  = f"{BASE}/train-20260302T182945Z-3-001/train/train.csv"

EVAL_IMGDIR     = f"{BASE}/eval-20260302T182800Z-3-001/eval/val_no_labels/val_images"
EVAL_LABEL_CSV  = f"{BASE}/eval-20260302T182800Z-3-001/eval/index_label_val.csv"
EVAL_TEXT_CSV   = f"{BASE}/eval-20260302T182800Z-3-001/eval/val.csv"

TEST_IMGDIR     = f"{BASE}/test_release-20260302T183217Z-3-001/test_release/test_images"
TEST_TEXT_CSV   = f"{BASE}/test_release-20260302T183217Z-3-001/test_release/index_text.csv"

os.makedirs(f"{WORK_DIR}/csvs", exist_ok=True)
print("✓ Paths set")

✓ Paths set


In [3]:
import pandas as pd

train_label = pd.read_csv(TRAIN_LABEL_CSV)
train_text  = pd.read_csv(TRAIN_TEXT_CSV)
eval_label  = pd.read_csv(EVAL_LABEL_CSV)
eval_text   = pd.read_csv(EVAL_TEXT_CSV)
test_text   = pd.read_csv(TEST_TEXT_CSV)

print("=== TRAIN label CSV ===")
print(f"  Columns: {train_label.columns.tolist()}  Shape: {train_label.shape}")
print("\n=== TRAIN text CSV ===")
print(f"  Columns: {train_text.columns.tolist()}  Shape: {train_text.shape}")
print("\n=== EVAL label CSV ===")
print(f"  Columns: {eval_label.columns.tolist()}  Shape: {eval_label.shape}")
print("\n=== TEST text CSV ===")
print(f"  Columns: {test_text.columns.tolist()}  Shape: {test_text.shape}")
print("\n✓ CSVs loaded")

=== TRAIN label CSV ===
  Columns: ['index', 'label']  Shape: (8195, 2)

=== TRAIN text CSV ===
  Columns: ['index', 'label', 'post_text', 'image_text']  Shape: (8195, 4)

=== EVAL label CSV ===
  Columns: ['index', 'label']  Shape: (1024, 2)

=== TEST text CSV ===
  Columns: ['index', 'post_text', 'image_text']  Shape: (1025, 3)

✓ CSVs loaded


In [4]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TWEET_COL   = "post_text"
OCR_COL     = "image_text"
LABEL_NAMES = ["Vaccine-Critical", "Neutral", "Pro-Vaccine"]
NUM_CLASSES = 3

train_df = train_label[["index", "label"]].copy()
train_df["label"] = train_df["label"].astype(int)

eval_df = eval_label[["index", "label"]].copy()
eval_df["label"] = eval_df["label"].astype(int)

print(f"Train: {len(train_df)} rows")
print(train_df["label"].value_counts().rename(
    {0:"Vaccine-Critical",1:"Neutral",2:"Pro-Vaccine"}).to_string())
print(f"\nEval : {len(eval_df)} rows")
print(eval_df["label"].value_counts().rename(
    {0:"Vaccine-Critical",1:"Neutral",2:"Pro-Vaccine"}).to_string())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df["label"].values
)
cw_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"\nClass weight tensor: {cw_tensor}")
print(f"Class weights (balanced): {np.round(class_weights, 4)}")
print("  (higher weight = model penalised more for missing that class)")
print("✓ Labels normalised")

Train: 8195 rows
label
Pro-Vaccine         3199
Vaccine-Critical    2535
Neutral             2461

Eval : 1024 rows
label
Pro-Vaccine         389
Neutral             327
Vaccine-Critical    308

Class weight tensor: tensor([1.0776, 1.1100, 0.8539], device='cuda:0')
Class weights (balanced): [1.0776 1.11   0.8539]
  (higher weight = model penalised more for missing that class)
✓ Labels normalised


In [5]:
LABEL_IMGDIR = f"{BASE}/train-20260302T182945Z-3-001/train/images_by_label"
if os.path.exists(LABEL_IMGDIR):
    for cls_folder in sorted(os.listdir(LABEL_IMGDIR)):
        cls_path = os.path.join(LABEL_IMGDIR, cls_folder)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            print(f"  {cls_folder}: {count} images")
else:
    print("Label image dir not found — skipping")

print(f"\n  Train images : {len(os.listdir(TRAIN_IMGDIR))} files")
print(f"  Eval  images : {len(os.listdir(EVAL_IMGDIR))} files")
print(f"  Test  images : {len(os.listdir(TEST_IMGDIR))} files")

  anti_vaccine: 2535 images
  neutral: 2461 images
  pro_vaccine: 3199 images

  Train images : 8195 files
  Eval  images : 1024 files
  Test  images : 1025 files


In [6]:
def make_text_lookup(df, tweet_col, ocr_col=None):
    lookup = {}
    for _, r in df.iterrows():
        raw_idx = str(r["index"]).strip()
        idx = raw_idx if raw_idx.endswith(".png") else raw_idx + ".png"

        tweet = str(r.get(tweet_col, "") or "").strip()
        ocr   = str(r.get(ocr_col,   "") or "").strip() if ocr_col else ""
        tweet = "" if tweet.lower() == "nan" else tweet
        ocr   = "" if ocr.lower()   == "nan" else ocr

        if tweet and ocr:
            lookup[idx] = f"{tweet} [SEP] {ocr}"
        elif tweet:
            lookup[idx] = tweet
        elif ocr:
            lookup[idx] = ocr
        else:
            lookup[idx] = ""
    return lookup

train_text_lookup = make_text_lookup(train_text, TWEET_COL, OCR_COL)
eval_text_lookup  = make_text_lookup(eval_text,  TWEET_COL, OCR_COL)
test_text_lookup  = make_text_lookup(test_text,  TWEET_COL, OCR_COL)

for name, lookup in [("train", train_text_lookup),
                      ("eval",  eval_text_lookup),
                      ("test",  test_text_lookup)]:
    empty = sum(1 for v in lookup.values() if not v)
    print(f"  {name}: {len(lookup)} entries  |  empty: {empty}")
print("✓ Text lookups ready")

  train: 8195 entries  |  empty: 0
  eval: 1024 entries  |  empty: 0
  test: 1025 entries  |  empty: 0
✓ Text lookups ready


In [7]:
def build_model_csv(label_df, img_dir, text_lookup, out_path, is_test=False):
    rows = []
    if is_test:
        for fname in sorted(f for f in os.listdir(img_dir)
                            if f.lower().endswith((".jpg",".jpeg",".png",".webp"))):
            rows.append({
                "index":      fname,
                "image_path": os.path.join(img_dir, fname),
                "text":       text_lookup.get(fname, ""),
            })
    else:
        for _, row in label_df.iterrows():
            raw = str(row["index"]).strip()
            fname = raw if raw.endswith(".png") else raw + ".png"
            rows.append({
                "index":      fname,
                "image_path": os.path.join(img_dir, fname),
                "text":       text_lookup.get(fname, ""),
                "label":      int(row["label"]),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return out_path

train_csv_path = f"{WORK_DIR}/csvs/train.csv"
val_csv_path   = f"{WORK_DIR}/csvs/val.csv"
test_csv_path  = f"{WORK_DIR}/csvs/test.csv"

build_model_csv(train_df, TRAIN_IMGDIR, train_text_lookup, train_csv_path)
build_model_csv(eval_df,  EVAL_IMGDIR,  eval_text_lookup,  val_csv_path)
build_model_csv(None,     TEST_IMGDIR,  test_text_lookup,  test_csv_path, is_test=True)

for name, path in [("train", train_csv_path),
                    ("val",   val_csv_path),
                    ("test",  test_csv_path)]:
    df = pd.read_csv(path)
    print(f"  {name}.csv: {len(df)} rows")
print("✓ CSVs ready")

  train.csv: 8195 rows
  val.csv: 1024 rows
  test.csv: 1025 rows
✓ CSVs ready


In [8]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
import numpy as np, warnings, time
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

Device : cuda
GPU    : Tesla T4


In [9]:
from PIL import Image
from torchvision import transforms

class DINOv2Dataset(Dataset):
    """DINOv2 image-only — uses AutoImageProcessor for ViT-compatible preprocessing."""
    def __init__(self, csv_path, processor, is_test=False):
        self.df        = pd.read_csv(csv_path)
        self.processor = processor
        self.is_test   = is_test
        self.augment   = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.4),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomAffine(degrees=10, translate=(0.08, 0.08)),
        ]) if not is_test else None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = Image.open(str(row["image_path"])).convert("RGB")
            if self.augment: image = self.augment(image)
        except Exception:
            image = Image.new("RGB", (224, 224), (128, 128, 128))

        enc  = self.processor(images=image, return_tensors="pt")
        item = {
            "index":        str(row["index"]),
            "pixel_values": enc["pixel_values"].squeeze(0),
        }
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item


class TextOnlyDataset(Dataset):
    """Text only — RoBERTa-large."""
    def __init__(self, csv_path, tokenizer, max_text_len=256, is_test=False):
        self.df           = pd.read_csv(csv_path)
        self.tokenizer    = tokenizer
        self.max_text_len = max_text_len
        self.is_test      = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row.get("text", "") or "").strip()
        text = "[empty]" if not text or text.lower() == "nan" else text
        enc  = self.tokenizer(
            text, max_length=self.max_text_len,
            padding="max_length", truncation=True, return_tensors="pt")
        item = {
            "index":          str(row["index"]),
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

print("✓ DINOv2Dataset and TextOnlyDataset defined")

✓ DINOv2Dataset and TextOnlyDataset defined


In [10]:
from transformers import AutoModel, AutoTokenizer

class TextOnlyModel(nn.Module):
    """RoBERTa-large text only — unchanged from Model 3."""
    def __init__(self, model_name, num_classes=3, dropout=0.3, freeze=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),    nn.GELU(), nn.Dropout(dropout * 0.7),
            nn.Linear(256, num_classes)
        )

    def unfreeze_top_layers(self, n=4):
        for layer in list(self.encoder.encoder.layer)[-n:]:
            for p in layer.parameters(): p.requires_grad = True

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :].float()
        return self.classifier(cls)


class DINOv2ImageModel(nn.Module):
    """DINOv2-base image-only classifier.
    - CLS token from last hidden state → classifier head
    - Freeze by default; unfreeze top n layers on schedule
    """
    def __init__(self, model_name="facebook/dinov2-base",
                 num_classes=3, dropout=0.3, freeze=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
        hidden = self.encoder.config.hidden_size   # 768 for dinov2-base
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),    nn.GELU(), nn.Dropout(dropout * 0.7),
            nn.Linear(256, num_classes)
        )

    def unfreeze_top_layers(self, n=4):
        layers = list(self.encoder.encoder.layer)
        for layer in layers[-n:]:
            for p in layer.parameters(): p.requires_grad = True
        print(f"  Trainable: {sum(p.numel() for p in self.parameters() if p.requires_grad):,}")

    def forward(self, pixel_values):
        out = self.encoder(pixel_values=pixel_values)
        cls = out.last_hidden_state[:, 0, :].float()
        return self.classifier(cls)

print("✓ TextOnlyModel and DINOv2ImageModel defined")

✓ TextOnlyModel and DINOv2ImageModel defined


In [11]:
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup
import time

class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes=NUM_CLASSES, smoothing=0.1):
        super().__init__()
        self.smoothing, self.cls = smoothing, classes
    def forward(self, logits, targets):
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.cls - 1)
        with torch.no_grad():
            st = torch.full_like(logits, smooth_val)
            st.scatter_(1, targets.unsqueeze(1), confidence)
        return -(st * F.log_softmax(logits, -1)).sum(-1).mean()

class WeightedLabelSmoothingLoss(nn.Module):
    def __init__(self, class_weights, classes=NUM_CLASSES, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.cls       = classes
        self.register_buffer("weights", class_weights)
    def forward(self, logits, targets):
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.cls - 1)
        with torch.no_grad():
            st = torch.full_like(logits, smooth_val)
            st.scatter_(1, targets.unsqueeze(1), confidence)
        log_prob      = F.log_softmax(logits, -1)
        sample_loss   = -(st * log_prob).sum(-1)
        sample_weight = self.weights[targets]
        return (sample_loss * sample_weight).mean()

scalers = {
    "text": torch.amp.GradScaler('cuda'),
    "dino": torch.amp.GradScaler('cuda'),
}

def train_one_epoch(model, loader, opt, sched, criterion, mode="dino"):
    model.train()
    scaler = scalers[mode]
    total_loss, all_preds, all_labels = 0.0, [], []
    for batch in tqdm(loader, desc="  Train", leave=False):
        labels = batch["label"].to(DEVICE)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            if mode == "text":
                logits = model(batch["input_ids"].to(DEVICE),
                               batch["attention_mask"].to(DEVICE))
            else:
                logits = model(batch["pixel_values"].to(DEVICE))
            loss = criterion(logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            opt.zero_grad(); continue

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        sched.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return total_loss / len(loader), f1_score(
        all_labels, all_preds, average="macro", zero_division=0)


@torch.no_grad()
def evaluate(model, loader, mode="dino"):
    model.eval()
    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc="  Val  ", leave=False):
        with torch.amp.autocast('cuda'):
            if mode == "text":
                logits = model(batch["input_ids"].to(DEVICE),
                               batch["attention_mask"].to(DEVICE))
            else:
                logits = model(batch["pixel_values"].to(DEVICE))
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(batch["label"].tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    report   = classification_report(
        all_labels, all_preds, target_names=LABEL_NAMES, zero_division=0)
    return macro_f1, report


def run_training(model, train_loader, val_loader, cfg, mode="dino", cw_tensor=None):
    criterion = (WeightedLabelSmoothingLoss(cw_tensor)
                 if cw_tensor is not None else LabelSmoothingLoss())

    if cfg.get("unfreeze_epoch", 999) == 0:
        enc_params, head_params = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad: continue
            is_enc = any(k in name for k in ["encoder"])
            (enc_params if is_enc else head_params).append(p)
        optimizer = torch.optim.AdamW([
            {"params": enc_params,  "lr": 2e-6},
            {"params": head_params, "lr": cfg["lr"]},
        ], weight_decay=cfg["weight_decay"])
    else:
        params    = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = torch.optim.AdamW(
            params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    steps     = len(train_loader) * cfg["epochs"]
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(0.1 * steps), steps)

    best_f1, history = 0.0, []
    print(f"\n{'Ep':>3}  {'Loss':>7}  {'TrainF1':>8}  {'ValF1':>7}  {'Best':>5}  {'Sec':>5}")
    print("─" * 46)

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        if epoch == cfg.get("unfreeze_epoch", 999):
            print(f"\n  ── Epoch {epoch}: unfreezing top 4 layers ──")
            model.unfreeze_top_layers(4)
            enc_params, head_params = [], []
            for name, p in model.named_parameters():
                if not p.requires_grad: continue
                is_enc = "encoder" in name
                (enc_params if is_enc else head_params).append(p)
            optimizer = torch.optim.AdamW([
                {"params": enc_params,  "lr": 2e-6},
                {"params": head_params, "lr": cfg["lr"]},
            ], weight_decay=cfg["weight_decay"])
            remaining = len(train_loader) * (cfg["epochs"] - epoch + 1)
            scheduler = get_cosine_schedule_with_warmup(
                optimizer, int(0.1 * remaining), remaining)
            scalers[mode] = torch.amp.GradScaler('cuda')

        loss, tr_f1    = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion, mode)
        val_f1, report = evaluate(model, val_loader, mode)
        elapsed        = time.time() - t0
        is_best        = val_f1 > best_f1

        if is_best:
            best_f1 = val_f1
            torch.save(model.state_dict(), cfg["save_path"])

        history.append({"epoch": epoch, "loss": loss,
                         "train_f1": tr_f1, "val_f1": val_f1})
        print(f"  {epoch:2d}   {loss:7.4f}   {tr_f1:8.4f}   {val_f1:7.4f}"
              f"   {'★' if is_best else ' '}    {elapsed:4.0f}s")
        if is_best: print(f"\n{report}\n")

    print(f"\n✓ Best Val Macro-F1: {best_f1:.4f} → {cfg['save_path']}")
    return best_f1, history

print("✓ Training functions defined (text + dino modes)")

✓ Training functions defined (text + dino modes)


In [12]:
df = pd.read_csv(f"{WORK_DIR}/csvs/train.csv")
print(f"Columns: {df.columns.tolist()}")
print(f"Empty text: {(df['text'].fillna('').str.strip() == '').sum()} / {len(df)}")
print(f"\nFirst 5 rows:")
for _, r in df.head(5).iterrows():
    print(f"  label={r['label']}  text='{str(r['text'])[:100]}'")

print("\n--- Source train.csv ---")
src = pd.read_csv(TRAIN_TEXT_CSV)
print(f"Columns: {src.columns.tolist()}")
print(f"post_text empty: {src['post_text'].isna().sum()} / {len(src)}")
print(f"image_text empty: {src['image_text'].isna().sum()} / {len(src)}")
print("\n✓ Text lookups ready")

Columns: ['index', 'image_path', 'text', 'label']
Empty text: 0 / 8195

First 5 rows:
  label=2  text='Got my second shot, fully vaxxed, baby! It went so much faster, there were a lot fewer people in lin'
  label=2  text='With ???@RimaBrusi???  - All #vaxxed up and ready to go! https://t.co/tpkMXIRwxA'
  label=2  text='This beautiful frontline RN just got vaxxed! I?????m so happy ?????? @lynne_lisa #COVID19Vaccine #Va'
  label=0  text='In Kenya, the catholic church unearthed an ongoing campaign by globalists to sterilize women of repr'
  label=2  text='Excited to be vaxxed! #fauciouchie https://t.co/38dWULIuMV [SEP] covin19Cay ChoygtFac Nana Malar Tea'

--- Source train.csv ---
Columns: ['index', 'label', 'post_text', 'image_text']
post_text empty: 0 / 8195
image_text empty: 3411 / 8195

✓ Text lookups ready


In [15]:
TEXT_CFG = {
    "model_name":     "roberta-large",
    "max_text_len":   256,
    "batch_size":     8,
    "epochs":         8,
    "lr":             2e-5,
    "weight_decay":   1e-2,
    "unfreeze_epoch": 999,
    "save_path":      f"{WORK_DIR}/best_text_model.pt",
}

print("=== EXPERIMENT 1: Text Only (RoBERTa-large) ===")
text_tok = AutoTokenizer.from_pretrained(TEXT_CFG["model_name"])

train_text_ds = TextOnlyDataset(train_csv_path, text_tok, TEXT_CFG["max_text_len"])
val_text_ds   = TextOnlyDataset(val_csv_path,   text_tok, TEXT_CFG["max_text_len"])

train_text_loader = DataLoader(
    train_text_ds, batch_size=TEXT_CFG["batch_size"],
    shuffle=True,  num_workers=0, pin_memory=True)
val_text_loader = DataLoader(
    val_text_ds,   batch_size=TEXT_CFG["batch_size"],
    shuffle=False, num_workers=0, pin_memory=True)

text_model = TextOnlyModel(
    TEXT_CFG["model_name"], dropout=0.4, freeze=False).to(DEVICE)

print(f"Total params    : {sum(p.numel() for p in text_model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in text_model.parameters() if p.requires_grad):,}")

text_f1, text_history = run_training(
    text_model, train_text_loader, val_text_loader,
    TEXT_CFG, mode="text", cw_tensor=cw_tensor)

=== EXPERIMENT 1: Text Only (RoBERTa-large) ===


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Val  :   0%|          | 0/128 [00:00<?, ?it/s]

   7    0.5024     0.9479    0.7839         456s


  Train:   0%|          | 0/1025 [00:00<?, ?it/s]

  Val  :   0%|          | 0/128 [00:00<?, ?it/s]

   8    0.4867     0.9556    0.7823         456s

✓ Best Val Macro-F1: 0.8096 → /kaggle/working/best_text_model.pt


In [14]:
DINO_NAME = "facebook/dinov2-base"

DINO_CFG = {
    "model_name":     DINO_NAME,
    "batch_size":     16,
    "epochs":         8,
    "lr":             2e-5,
    "weight_decay":   1e-2,
    "save_path":      f"{WORK_DIR}/best_dino_model.pt",
}

print("=== EXPERIMENT 2: Image Only (DINOv2-base) ===")
dino_processor = AutoImageProcessor.from_pretrained(DINO_NAME)

train_dino_ds = DINOv2Dataset(train_csv_path, dino_processor, is_test=False)
val_dino_ds   = DINOv2Dataset(val_csv_path,   dino_processor, is_test=False)

train_dino_loader = DataLoader(
    train_dino_ds, batch_size=DINO_CFG["batch_size"],
    shuffle=True,  num_workers=0, pin_memory=True)
val_dino_loader = DataLoader(
    val_dino_ds,   batch_size=DINO_CFG["batch_size"],
    shuffle=False, num_workers=0, pin_memory=True)

dino_model = DINOv2ImageModel(
    DINO_NAME, num_classes=NUM_CLASSES,
    dropout=0.3, freeze=False).to(DEVICE)

print(f"Total params    : {sum(p.numel() for p in dino_model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in dino_model.parameters() if p.requires_grad):,}")

dino_f1, dino_history = run_training(
    dino_model, train_dino_loader, val_dino_loader,
    DINO_CFG, mode="dino", cw_tensor=cw_tensor)

=== EXPERIMENT 2: Image Only (DINOv2-base) ===


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Total params    : 87,107,331
Trainable params: 87,107,331

 Ep     Loss   TrainF1    ValF1   Best    Sec
──────────────────────────────────────────────


  Train:   0%|          | 0/513 [00:00<?, ?it/s]

  Val  :   0%|          | 0/64 [00:00<?, ?it/s]

...

  Val  :   0%|          | 0/64 [00:00<?, ?it/s]

   7    0.5975     0.8671    0.6954         459s


  Train:   0%|          | 0/513 [00:00<?, ?it/s]

  Val  :   0%|          | 0/64 [00:00<?, ?it/s]

   8    0.5618     0.8921    0.6889         461s

✓ Best Val Macro-F1: 0.6988 → /kaggle/working/best_dino_model.pt


In [16]:
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support, classification_report
from itertools import product as iproduct
from tqdm.auto import tqdm

@torch.no_grad()
def get_softmax_probs(model, loader, mode="dino"):
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    for batch in tqdm(loader, desc=f"  Probs/{mode}", leave=False):
        with torch.amp.autocast('cuda'):
            if mode == "text":
                logits = model(batch["input_ids"].to(DEVICE),
                               batch["attention_mask"].to(DEVICE))
            else:
                logits = model(batch["pixel_values"].to(DEVICE))
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        if "label" in batch:
            all_labels.extend(batch["label"].tolist())
        if "index" in batch:
            all_indices.extend(batch["index"])
    return np.concatenate(all_probs, axis=0), all_labels, all_indices

text_model.load_state_dict(torch.load(TEXT_CFG["save_path"], map_location=DEVICE))
dino_model.load_state_dict(torch.load(DINO_CFG["save_path"], map_location=DEVICE))

print("Computing softmax probabilities on validation set...")
probs_text, true_labels, _ = get_softmax_probs(text_model, val_text_loader, "text")
probs_dino, _,           _ = get_softmax_probs(dino_model, val_dino_loader, "dino")

true_labels = np.array(true_labels)

f1_txt  = f1_score(true_labels, probs_text.argmax(1), average="macro", zero_division=0)
f1_dino = f1_score(true_labels, probs_dino.argmax(1), average="macro", zero_division=0)

p1, r1, _, _ = precision_recall_fscore_support(
    true_labels, probs_text.argmax(1), average="macro", zero_division=0)
p2, r2, _, _ = precision_recall_fscore_support(
    true_labels, probs_dino.argmax(1), average="macro", zero_division=0)

print("\n" + "="*62)
print(f"{'Model':<20} {'Classifier':<22} {'P':>6} {'R':>6} {'F1':>6}")
print("="*62)
print(f"{'Unimodal (Text)':<20} {'RoBERTa-large':<22} {p1:>6.4f} {r1:>6.4f} {f1_txt:>6.4f}")
print(f"{'Unimodal (Image)':<20} {'DINOv2-base':<22} {p2:>6.4f} {r2:>6.4f} {f1_dino:>6.4f}")
print("="*62)

total = f1_txt + f1_dino
w_t   = f1_txt  / total
w_d   = f1_dino / total
print(f"\nEnsemble weights → text:{w_t:.3f}  dino:{w_d:.3f}")

ensemble_probs = w_t * probs_text + w_d * probs_dino
ensemble_preds = ensemble_probs.argmax(1)
f1_ens = f1_score(true_labels, ensemble_preds, average="macro", zero_division=0)
print(f"\n{'='*62}")
print(f"Weighted Ensemble Macro-F1: {f1_ens:.4f}")
print("="*62)
print(classification_report(true_labels, ensemble_preds,
                             target_names=LABEL_NAMES, zero_division=0))

def apply_thresholds(probs, thresholds):
    return (probs / np.array(thresholds)).argmax(1)

best_thresh_f1  = f1_ens
best_thresholds = [1.0, 1.0, 1.0]
threshold_vals  = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]

print("\nSearching best per-class thresholds...")
for t0, t1, t2 in iproduct(threshold_vals, threshold_vals, threshold_vals):
    preds = apply_thresholds(ensemble_probs, [t0, t1, t2])
    f1    = f1_score(true_labels, preds, average="macro", zero_division=0)
    if f1 > best_thresh_f1:
        best_thresh_f1  = f1
        best_thresholds = [t0, t1, t2]

print(f"Best thresholds : {best_thresholds}")
print(f"Threshold-tuned Ensemble F1: {best_thresh_f1:.4f}  (raw: {f1_ens:.4f})")
print(classification_report(
    true_labels, apply_thresholds(ensemble_probs, best_thresholds),
    target_names=LABEL_NAMES, zero_division=0))

Computing softmax probabilities on validation set...


  Probs/text:   0%|          | 0/128 [00:00<?, ?it/s]

  Probs/dino:   0%|          | 0/64 [00:00<?, ?it/s]


Model                Classifier                  P      R     F1
Unimodal (Text)      RoBERTa-large          0.8136 0.8084 0.8096
Unimodal (Image)     DINOv2-base            0.7033 0.7045 0.7034

Ensemble weights → text:0.535  dino:0.465

Weighted Ensemble Macro-F1: 0.8128
                  precision    recall  f1-score   support

Vaccine-Critical       0.82      0.83      0.82       308
         Neutral       0.75      0.76      0.75       327
     Pro-Vaccine       0.87      0.86      0.86       389

        accuracy                           0.82      1024
       macro avg       0.81      0.81      0.81      1024
    weighted avg       0.82      0.82      0.82      1024


Searching best per-class thresholds...
Best thresholds : [1.2, 1.1, 1.3]
Threshold-tuned Ensemble F1: 0.8220  (raw: 0.8128)
                  precision    recall  f1-score   support

Vaccine-Critical       0.84      0.82      0.83       308
         Neutral       0.73      0.80      0.76       327
     Pro-Vaccine

In [17]:
import zipfile

test_text_ds = TextOnlyDataset(test_csv_path, text_tok,
                                TEXT_CFG["max_text_len"], is_test=True)
test_dino_ds = DINOv2Dataset(test_csv_path, dino_processor, is_test=True)

test_text_loader = DataLoader(test_text_ds, batch_size=32, shuffle=False, num_workers=0)
test_dino_loader = DataLoader(test_dino_ds, batch_size=32, shuffle=False, num_workers=0)

print("Running test inference...")
test_probs_text, _, test_indices = get_softmax_probs(text_model, test_text_loader, "text")
test_probs_dino, _, _            = get_softmax_probs(dino_model, test_dino_loader, "dino")

test_ensemble = w_t * test_probs_text + w_d * test_probs_dino
test_preds    = apply_thresholds(test_ensemble, best_thresholds)

pred_df = pd.DataFrame({"index": test_indices, "label": test_preds})
pred_df["_key"] = pred_df["index"].str.extract(r"(\d+)").astype(int)
pred_df = pred_df.sort_values("_key").drop(columns="_key").reset_index(drop=True)

CSV_PATH = f"{WORK_DIR}/predictions.csv"
ZIP_PATH = f"{WORK_DIR}/submission.zip"
pred_df.to_csv(CSV_PATH, index=False)

df  = pd.read_csv(CSV_PATH)
ok  = (set(df.columns) == {"index", "label"} and
       not df.isnull().any().any() and
       df["label"].isin({0, 1, 2}).all() and
       not df["index"].duplicated().any() and
       df["index"].str.extract(r"(\d+)")[0].astype(int).is_monotonic_increasing)

if ok:
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(CSV_PATH, arcname="predictions.csv")
    print(f"✓ submission.zip ready ({os.path.getsize(ZIP_PATH)/1024:.1f} KB)")
    print(f"  Best Val Ensemble F1 : {best_thresh_f1:.4f}")
    print(f"  Text model F1        : {text_f1:.4f}")
    print(f"  DINOv2 model F1      : {dino_f1:.4f}")
    print("→ Download from Kaggle Output tab → upload to Codabench")
else:
    print("✗ Validation failed — check predictions")

Running test inference...


  Probs/text:   0%|          | 0/33 [00:00<?, ?it/s]

  Probs/dino:   0%|          | 0/33 [00:00<?, ?it/s]

✓ submission.zip ready (2.9 KB)
  Best Val Ensemble F1 : 0.8220
  Text model F1        : 0.8096
  DINOv2 model F1      : 0.6988
→ Download from Kaggle Output tab → upload to Codabench
